In [ ]:
Content
1. Web Session
2. Serialization
3. CORS

In [ ]:
# 1. Web Session
Why web session are required?
- HTTP is stateless, means each request are treated independently.
   But websites offen need to track user state, if user is logged in or not.

HTTP is stateless
• HTTP does not retain memory of previous requests
• Each request must contain all necessary information
• Why this is a challenge for user sessions

Ways to maintain Session
1. Session Based Authentication
    - The server stores session data, and the client holds only a session ID (usually in a cookie).
    - Example: Traditional Web Apps (Django, Rails, PHP), Server-side rendered apps with login

                                                                                  
    
Workflow:
    1. Client logs in via POST /login
    2. Server validates credentials
    3. Server creates a session (stores it in memory or DB)
    4. Server sends a session ID in a cookie to the client
    5. Client sends this cookie in every request
    6. On server side:
        - Server extracts session_id from the cookie
        - Server looks up session storage (e.g., Redis, DB, in-memory)
            - session = redis.get("session:abc123xyz")
            - If found, then it contains
                {
                  "user_id": 42,
                  "email": "sourav@example.com",
                  "role": "admin",
                  "expires": "2025-06-18T23:59:00Z"
                }

    6. Server uses session ID to retrieve user state
    Example: Session entry in DB session_id=abc123 → user_id=42, role='admin'

Note: Cookies are usually set by the server using the Set-Cookie header, and the browser automatically sends them back in subsequent requests. 
This is what powers session-based authentication. Security flags like HttpOnly and SameSite help protect against XSS and CSRF attacks.
    
Pros
    - Secure (server controls session)
    - Minimal client logic
    - Easy to manage permissions

Cons
    - Scaling is a problem: Session needs to be stored/shared (memcached, DB, Redis)
    - Require server-side cleanup for Ideal session.


Security Concerns:
    - Session Hijacking: Stolen session IDs
    - Cross-Site Request Forgery (CSRF): Unauthorized actions
    - Secure Cookie Handling: Avoiding theft via Secure, HttpOnly, and SameSite flags

How to achieve this on Server:
    - Sticky Session: Routing same user to the same server everytime.
    - Distributed Session: Useing a common redis.
        


2. Token Based Authentication (JWT, OAuth Tokens)
    - The client holds the token, and full session details are embeded in a token (like a JWT).
    - Server does not store the token/ session details.                                                                                       
    - The server is stateless — it just validates the token.
    - Example: Web apps (React, Angular), Microservices API Gateway auth

Workflow:
    1. Client logs in via POST /login
    2. Server generates a signed token (e.g., JWT)
    3. Server sends the token to the client
    4. Client stores it (localStorage, cookie, etc.)
    5. Client sends it in Authorization header on every request
    6. Server validates the token without session storage

Pros:
    - Stateless (scalable): No server memory or DB needed
    - Easy to use in APIs
    - Token has an expiry

Cons:
    - Hard to revoke: Token remains valid untill expired
    - Bigger payload: JWT are heavy
    - Sensitive if stored in localStorage: Needs CSRF/XSS protection


| Feature              | Session-Based              | Token-Based (JWT/OAuth)     |
|----------------------|----------------------------|-----------------------------|
| State stored         | Server-side (Redis, DB)    | Client-side (inside token)  |
| Scales across servers| ❌ Needs sticky sessions or shared state | ✅ Stateless |
| Token storage        | Cookie (HttpOnly)          | Header / Cookie / LocalStorage |
| Revocable            | ✅ Easily (delete session)  | ❌ Hard unless using blacklist |
| Expiry               | Configurable via session TTL | Included in token (exp)     |
| Use case             | Web apps (monolith)        | APIs, SPAs, Mobile          |

In [ ]:
# 2. Token Based Authentication

1. JWT (JSON Web Token)
    - JWT (JSON Web Token) is a compact, self-contained token that encodes a user’s session or identity and can be verified and trusted via a digital signature.

Components
<Header>.<Payload>.<Signature>
	1.	Header – Algorithm & token type
	2.	Payload – Claims (user data, roles, expiry)
	3.	Signature – HMAC SHA256 or RSA signature        

Request Header:
    Authorization: Bearer <token>

Pros:
    - Stateless: Server doesn’t store sessions
    - Portable: Can be used across services/domains
    - Self-contained: Includes user data and expiry

Cons:
    - Token revocation: Hard to invalidate before expiry
    - Size: Bigger than session IDs
    - Storage security: XSS if stored in localStorage

Can user tamper the token and edit the role?
    - The server will recalculate the signature using the original secret key, and it will not match the tampered token’s signature.
    - When server receives the token, it recalculates the token using a SECRET_KEY private key and it is only known to the server. Client can not forge the token without the key.
    - Code
        signature = sign(base64(header) + "." + base64(payload), SECRET_KEY)
        
        if incoming_signature != signature:
            raise InvalidTokenError()



In [ ]:
# Authorization Framework

1. OAuth 2.0 (Authorization Framework)
    - OAuth 2.0 is not a token format, but a protocol that allows one application to access another’s resources on behalf of a user.
    - Ex: A travel app wants to access your Google Calendar.

[User] ---> [App] ---redirects---> [Google OAuth Server]
                                    |
                             User Logs In & Approves
                                    |
          [App] <--- auth code ----/
          [App] --(code)--> [Google Token Endpoint]
                              |
                      <- access_token, id_token

Access Token – used to access APIs
	•	ID Token – contains user identity (like a JWT)
	•	Refresh Token – used to get a new access token        


In [ ]:
# Serialization and Deserialization

Serialization:
    Converting an in-memory object (e.g., a Python dict or Java object) into a format (like JSON, XML, binary) so it can be saved to disk or transmitted over a network.

Analogy
    Think of serialization like packaging a product for delivery.

Used in
    APIs, saving database

Example formats
    Json, XML, 
    Protocol Buffers (protobuf) (require pre-defined schema) used in gRPC - require predefined schema, smaller payload and faste processing.
        
import json
# Serialization
data = {"name": "Sourav", "role": "Engineer"}
json_str = json.dumps(data)  # '{"name": "Sourav", "role": "Engineer"}'

# Deserialization
parsed_data = json.loads(json_str)  # {'name': 'Sourav', 'role': 'Engineer'}


In [ ]:
CORS: Cross origin resouce sharing
Problem: Browser enforce same origin policy (SOP), and blocks the cross-origin requests by default.
    - If Web and backend APIs are hosted on different servers. 
    - It’s a browser-side security feature.

Request Flow:
1. Browser first sends a preflight request before making the actual requests (sent via Options request).
2. If the server responsds with the valid CORS headers (allowed method and headers).
3. Then the browser makes the actual request.

Solution 1:
    const cors = require("cors");
    app.use(cors({ origin: "http://localhost:3000" })); // allow only your frontend

Solution 2: 
Headers added on the server side.
app.use((req, res, next) => {
  res.header("Access-Control-Allow-Origin", "*"); // <-- Response header
  res.header("Access-Control-Allow-Headers", "Origin, X-Requested-With, Content-Type, Accept");
  next();
});

Handle Security Concerns
 - Always use a whitelist of trusted origins instead of *
 - Use Reverse Proxy and API gateway for CORS handling.

Alternatives to CORS for Cross-Origin Requests
    • Reverse Proxy:
    • Forwards client requests to the backend, bypassing browser CORS restrictions.
    • Example: Nginx proxying requests to a backend API.
    • API Gateway Handling CORS:
    • Centralized control of CORS policies for multiple services.
    • Example: AWS API Gateway configuring allowed, origins.

How API Gateways & Reverse Proxies Help?
    • Reverse Proxies handle cross-origin requests internally to avoid CORS issues.
    API Gateways enforce CORS policies centrally, ensuring security & consistency.
    Both improve performance by reducing unnecessary browser preflight requests.

In [ ]:
# Preflight Request

A preflight request is an automatic HTTP OPTIONS request sent by the browser before the actual request, to ask the server for permission to proceed with a cross-origin request

Hey server, is it okay if I send a POST request with these custom headers from another origin?
If the server says yes (by returning appropriate Access-Control-Allow-* headers), then the actual request is sent.
The server responds with the allowed method and headers.

Why it is required:
For security. It protects the server by giving it a chance to approve or deny requests that:
	•	Come from different origins,
	•	Use sensitive methods or headers,